# RoadGuard AI detection baseline

Pretrained YOLO11n · CUDA · Validation-selected checkpoint

Open the completed baseline and comparison here. The launch cell stays disabled when reopening archived results.

## 1. Select a dataset snapshot or an archived run

In [ ]:
import json

import pandas as pd
from IPython.display import Image, display

from roadguard.config import load_config
from roadguard.data.prepare import read_snapshot, snapshot_summary
from roadguard.training.archive import list_experiments, open_experiment
from roadguard.training.trainer import train
from roadguard.utils.device import environment_info, require_cuda
from roadguard.utils.paths import project_root

root = project_root()
profile = root / "configs/training/stage3_baseline.yaml"

# Windows notebooks use zero workers. The PowerShell CLI can use the YAML worker count.
overrides = ["training.workers=0"]
config = load_config(root, profile=profile, overrides=overrides)
snapshot = root / config["data"]["processed_root"] / config["data"]["snapshot_fingerprint"]

experiments = root / config["experiment"]["experiments_root"]
comparison_path = experiments / "stage3_comparison.json"
comparison = (
    json.loads(comparison_path.read_text(encoding="utf-8")) if comparison_path.is_file() else None
)
run_name = comparison["baseline"]["run"] if comparison else None
# Assign a different available run name here to inspect its saved results.
available_runs = list_experiments(experiments)
print("Available runs", available_runs)
if run_name is not None and run_name not in available_runs:
    raise ValueError("Choose one of the listed run directory names.")
archived = open_experiment(experiments / run_name) if run_name else None

## 2. Verify the GPU and CUDA kernels

In [ ]:
environment = environment_info()
display(
    pd.Series(
        {
            "GPU": environment["gpu"],
            "CUDA runtime": environment["cuda_runtime"],
            "CUDA available": environment["cuda_available"],
            "PyTorch": environment["torch"],
            "TorchVision": environment["torchvision"],
            "Ultralytics": environment["ultralytics"],
        },
        name="Current notebook kernel",
    )
)

try:
    device = require_cuda()
except RuntimeError as error:
    print(error)
else:
    print(f"{device}: tensor execution and TorchVision CUDA NMS passed.")

## 3. Dataset snapshot and statistics

Training and checkpoint selection use the snapshot’s train and validation partitions. The held-out partition is listed for provenance.

In [ ]:
if archived:
    fingerprint = archived.metadata["dataset_fingerprint"]
    splits = archived.metadata.get("dataset_summary", {})
elif snapshot:
    fingerprint = snapshot.name
    splits = snapshot_summary(read_snapshot(snapshot))
else:
    fingerprint = None
    splits = {}

print("Dataset fingerprint", fingerprint or "Pending Stage 2 preparation")
if splits:
    display(
        pd.DataFrame(
            {
                name: {key: row[key] for key in ("images", "objects") if key in row}
                for name, row in splits.items()
            }
        ).T
    )
    for name in ("train", "val"):
        if "annotations_per_class" in splits[name]:
            display(pd.Series(splits[name]["annotations_per_class"], name=f"{name} annotations"))

## 4. Training configuration

In [ ]:
settings = archived.metadata["config"] if archived else config
training = settings["training"]
display(
    pd.Series(
        {key: value for key, value in training.items() if key != "augmentation"},
        name="Training settings",
    )
)
display(pd.Series(training["augmentation"], name="Augmentations"))

## 5. Model configuration

In [ ]:
model = settings["model"]
weights = model["weights"] or f"yolo11{model['model_variant']}.pt"
display(pd.Series({**model, "weights": weights}, name="YOLO11 detection settings"))

## 6. Launch training

To launch a separate baseline run on the approved snapshot, clear the archived run selection and explicitly set the switch below. Use the PowerShell command for the same worker configuration as the recorded Stage 3 runs.

In [ ]:
launch_training = False

if launch_training:
    launch_training = False
    if archived is not None or snapshot is None:
        raise ValueError("Select a prepared snapshot and clear the archived run selection.")
    run = train(root, config, snapshot)
    archived = open_experiment(run)
    print(f"Saved experiment: {run}")
else:
    print(
        "Training launch is disabled."
        if archived is None
        else f"Reviewing {archived.directory.name}"
    )

## 7. Epoch-by-epoch metrics

In [ ]:
if archived and archived.epoch_metrics:
    history = pd.DataFrame(archived.epoch_metrics)
    from IPython.display import HTML

    table = history.to_html(index=False, classes="dataframe")
    display(HTML(f'<div style="max-height: 420px; overflow: auto">{table}</div>'))
else:
    print("Select a saved run with an epoch history.")

## 8. Training and validation curves

In [ ]:
def show_run_plots(*names):
    if archived:
        for file in archived.plots(*names):
            display(Image(filename=str(file)))


show_run_plots(
    "results.png", "BoxPR_curve.png", "BoxP_curve.png", "BoxR_curve.png", "BoxF1_curve.png"
)

## 9. Best checkpoint metrics

The best checkpoint is selected by validation mAP50–95. Precision and recall in the full validation report come from the validator’s operating point.

In [ ]:
metrics = archived.metadata.get("validation_metrics") if archived else None
if metrics:
    display(
        pd.Series(
            {
                "Precision": metrics["precision"],
                "Recall": metrics["recall"],
                "mAP50": metrics["map50"],
                "mAP50–95": metrics["map50_95"],
                "Macro F1": metrics["macro_f1"],
            },
            name="Best checkpoint · validation",
        )
    )
    display(
        pd.DataFrame.from_dict(
            {label: values or {} for label, values in metrics["per_class"].items()}, orient="index"
        )
        .reindex(metrics["per_class"])
        .rename_axis("Class")
    )
else:
    print("Validation metrics are available after a completed run.")

## 10. Confusion matrix

In [ ]:
show_run_plots("confusion_matrix.png", "confusion_matrix_normalized.png")

## 11. Best checkpoint information

In [ ]:
if archived and archived.best:
    display(pd.Series(archived.best, name="Validation-selected checkpoint"))
elif archived:
    print("This run has no archived checkpoint metadata.")

## 12. Sample predictions

Side-by-side labels and predictions are saved with the run. Reopening the notebook can display them without loading model weights.

In [ ]:
review = archived.validation_review if archived else None
if review:
    for sample in review["samples"][:3]:
        show_run_plots(sample["figure"])
else:
    print("Saved validation prediction examples are available after a completed run.")

## 13. Error analysis

These counts describe a reproducible validation sample at the saved confidence threshold and an IoU match threshold of 0.50. Compare misses, extra detections and class confusions with the full validation metrics.

In [ ]:
if review:
    print(f"{review['sample_count']} images at confidence ≥ {review['confidence_threshold']:.2f}")
    display(pd.Series(review["totals"], name="Sample error counts"))
    display(pd.DataFrame.from_dict(review["per_class"], orient="index"))
    if review["class_confusions"]:
        display(pd.Series(review["class_confusions"], name="Overlapping wrong-class predictions"))
    ranked = sorted(
        review["samples"],
        key=lambda row: row["false_positives"] + row["false_negatives"],
        reverse=True,
    )
    for sample in [row for row in ranked if row["false_positives"] or row["false_negatives"]][:3]:
        show_run_plots(sample["figure"])

## 14. Final experiment summary

In [ ]:
if archived:
    info = archived.metadata
    display(
        pd.Series(
            {
                "Run": archived.directory.name,
                "Status": info["status"],
                "Dataset": info["dataset_fingerprint"],
                "Training GPU": info.get("gpu"),
                "Epochs logged": len(archived.epoch_metrics),
                "Best validation mAP50–95": info.get("best_validation_metric"),
                "Validation review": info.get("validation_review_status"),
                "Artifact directory": str(archived.directory),
            },
            name="Experiment",
        )
    )
else:
    print("Select a completed Stage 3 run to display its summary.")

## 15. Matched validation comparison

Both models use the same frozen split and training budget. The selection criterion is validation mAP50–95.

In [ ]:
if comparison:
    columns = ["map50_95", "map50", "precision", "recall", "macro_f1"]
    table = {
        entry["run"]: {
            "best_epoch": entry["best_epoch"],
            **{metric: entry["validation_metrics"][metric] for metric in columns},
        }
        for entry in (comparison["baseline"], comparison["challenger"])
    }
    display(pd.DataFrame.from_dict(table, orient="index").round(4))
    display(
        pd.DataFrame(
            {
                entry["run"]: {
                    label: values["ap50_95"] if values else None
                    for label, values in entry["validation_metrics"]["per_class"].items()
                }
                for entry in (comparison["baseline"], comparison["challenger"])
            }
        )
        .round(4)
        .rename_axis("Class · validation AP50–95")
    )
    print("Validation selected", comparison["selected_run"])
else:
    print("The matched comparison appears after both validation runs finish.")